# Data Exploration - E-commerce Customer Future Value Prediction

This notebook performs exploratory data analysis on the cleaned UCI Online Retail II dataset.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import sys

# Add src to path
sys.path.append(str(Path.cwd().parent / 'src'))

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

In [ ]:
# Load cleaned data
from utils import get_processed_data_dir

processed_dir = get_processed_data_dir()
df = pd.read_csv(processed_dir / 'clean_transactions.csv', parse_dates=['invoicedate'])

print(f"Dataset shape: {df.shape}")
print(f"\nDate range: {df['invoicedate'].min()} to {df['invoicedate'].max()}")
print(f"\nUnique customers: {df['customerid'].nunique():,}")
print(f"Unique invoices: {df['invoiceno'].nunique():,}")
print(f"Unique products: {df['stockcode'].nunique():,}")
print(f"Unique countries: {df['country'].nunique()}")

In [ ]:
# Display sample data
df.head()

In [ ]:
# Data types and missing values
df.info()

## Customer Behavior Analysis

In [ ]:
# Customer-level statistics
customer_stats = df.groupby('customerid').agg({
    'invoiceno': 'nunique',
    'revenue': 'sum',
    'quantity': 'sum',
    'invoicedate': ['min', 'max']
}).reset_index()

customer_stats.columns = ['customerid', 'num_orders', 'total_revenue', 'total_quantity', 'first_purchase', 'last_purchase']
customer_stats['customer_tenure_days'] = (customer_stats['last_purchase'] - customer_stats['first_purchase']).dt.days + 1
customer_stats['avg_order_value'] = customer_stats['total_revenue'] / customer_stats['num_orders']

print("Customer Statistics Summary:")
customer_stats.describe()

In [ ]:
# Distribution of customers by order count
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Orders per customer
sns.histplot(customer_stats['num_orders'], bins=50, ax=axes[0,0])
axes[0,0].set_title('Distribution of Orders per Customer')
axes[0,0].set_xlabel('Number of Orders')
axes[0,0].set_ylabel('Number of Customers')
axes[0,0].set_xlim(0, 100)

# Revenue per customer (log scale)
sns.histplot(np.log1p(customer_stats['total_revenue']), bins=50, ax=axes[0,1])
axes[0,1].set_title('Distribution of Customer Revenue (Log Scale)')
axes[0,1].set_xlabel('Log(Revenue + 1)')
axes[0,1].set_ylabel('Number of Customers')

# Average order value
sns.histplot(customer_stats['avg_order_value'], bins=50, ax=axes[1,0])
axes[1,0].set_title('Distribution of Average Order Value')
axes[1,0].set_xlabel('Average Order Value (£)')
axes[1,0].set_ylabel('Number of Customers')
axes[1,0].set_xlim(0, 500)

# Customer tenure
sns.histplot(customer_stats['customer_tenure_days'], bins=50, ax=axes[1,1])
axes[1,1].set_title('Distribution of Customer Tenure (Days)')
axes[1,1].set_xlabel('Tenure (Days)')
axes[1,1].set_ylabel('Number of Customers')

plt.tight_layout()
plt.show()

In [ ]:
# Repeat vs one-time customers
repeat_customers = (customer_stats['num_orders'] > 1).sum()
one_time_customers = (customer_stats['num_orders'] == 1).sum()

print(f"Repeat customers: {repeat_customers} ({repeat_customers/len(customer_stats)*100:.1f}%)")
print(f"One-time customers: {one_time_customers} ({one_time_customers/len(customer_stats)*100:.1f}%)")

## Sales Behavior Analysis

In [ ]:
# Monthly revenue and orders
df['year_month'] = df['invoicedate'].dt.to_period('M')

monthly_stats = df.groupby('year_month').agg({
    'revenue': 'sum',
    'invoiceno': 'nunique',
    'customerid': 'nunique'
}).reset_index()

monthly_stats.columns = ['year_month', 'total_revenue', 'total_orders', 'unique_customers']
monthly_stats['avg_order_value'] = monthly_stats['total_revenue'] / monthly_stats['total_orders']

print("Monthly Statistics:")
monthly_stats.head(15)

In [ ]:
# Plot monthly trends
fig, axes = plt.subplots(3, 1, figsize=(15, 12))

# Monthly revenue
axes[0].plot(monthly_stats['year_month'].astype(str), monthly_stats['total_revenue'], marker='o')
axes[0].set_title('Monthly Revenue Over Time')
axes[0].set_xlabel('Month')
axes[0].set_ylabel('Revenue (£)')
axes[0].tick_params(axis='x', rotation=45)

# Monthly orders
axes[1].plot(monthly_stats['year_month'].astype(str), monthly_stats['total_orders'], marker='o', color='orange')
axes[1].set_title('Monthly Orders Over Time')
axes[1].set_xlabel('Month')
axes[1].set_ylabel('Number of Orders')
axes[1].tick_params(axis='x', rotation=45)

# Monthly unique customers
axes[2].plot(monthly_stats['year_month'].astype(str), monthly_stats['unique_customers'], marker='o', color='green')
axes[2].set_title('Monthly Unique Customers Over Time')
axes[2].set_xlabel('Month')
axes[2].set_ylabel('Number of Unique Customers')
axes[2].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# Average order value over time
plt.figure(figsize=(15, 5))
plt.plot(monthly_stats['year_month'].astype(str), monthly_stats['avg_order_value'], marker='o', color='purple')
plt.title('Average Order Value Over Time')
plt.xlabel('Month')
plt.ylabel('Average Order Value (£)')
plt.xticks(rotation=45)
plt.grid(True, alpha=0.3)
plt.show()

## Product Behavior Analysis

In [ ]:
# Product diversity per customer
product_diversity = df.groupby('customerid')['stockcode'].nunique().reset_index()
product_diversity.columns = ['customerid', 'unique_products']

print("Product Diversity Statistics:")
print(product_diversity['unique_products'].describe())

plt.figure(figsize=(10, 5))
sns.histplot(product_diversity['unique_products'], bins=50)
plt.title('Distribution of Unique Products per Customer')
plt.xlabel('Number of Unique Products')
plt.ylabel('Number of Customers')
plt.xlim(0, 200)
plt.show()

In [ ]:
# Top 10 most purchased products
top_products = df.groupby('description')['quantity'].sum().sort_values(ascending=False).head(10)

print("Top 10 Most Purchased Products:")
print(top_products)

plt.figure(figsize=(12, 6))
top_products.plot(kind='barh')
plt.title('Top 10 Most Purchased Products')
plt.xlabel('Total Quantity Sold')
plt.ylabel('Product Description')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## Cancellation/Return Behavior Analysis

In [ ]:
# Transaction type analysis
transaction_counts = df['transaction_type'].value_counts()
print("Transaction Types:")
print(transaction_counts)
print(f"\nCancellation rate: {(transaction_counts.get('cancellation', 0) / len(df) * 100):.2f}%")
print(f"Return rate: {(transaction_counts.get('return', 0) / len(df) * 100):.2f}%")

In [ ]:
# Customers with cancellations
cancellations = df[df['transaction_type'] == 'cancellation']
customers_with_cancellations = cancellations['customerid'].nunique()

print(f"Customers with cancellations: {customers_with_cancellations}")
print(f"Percentage of customers with cancellations: {customers_with_cancellations / df['customerid'].nunique() * 100:.1f}%")

# Cancellation revenue impact
cancellation_revenue = abs(cancellations['revenue'].sum())
total_revenue = df[df['transaction_type'] == 'normal']['revenue'].sum()
print(f"\nTotal cancellation revenue: £{cancellation_revenue:,.2f}")
print(f"Percentage of total revenue: {cancellation_revenue / total_revenue * 100:.2f}%")

In [ ]:
# Customer cancellation behavior
customer_cancellations = df[df['transaction_type'] == 'cancellation'].groupby('customerid').size().reset_index()
customer_cancellations.columns = ['customerid', 'cancellation_count']

print("Customers with High Cancellation Activity (5+ cancellations):")
high_cancellation = customer_cancellations[customer_cancellations['cancellation_count'] >= 5]
print(f"Count: {len(high_cancellation)}")
print(high_cancellations.sort_values('cancellation_count', ascending=False).head(10))

## Geographic Analysis

In [ ]:
# Revenue by country
country_stats = df.groupby('country').agg({
    'revenue': 'sum',
    'customerid': 'nunique',
    'invoiceno': 'nunique'
}).sort_values('revenue', ascending=False)

country_stats.columns = ['total_revenue', 'unique_customers', 'total_orders']
print("Revenue by Country (Top 10):")
print(country_stats.head(10))

In [ ]:
# Plot revenue by country (top 10)
plt.figure(figsize=(12, 6))
country_stats.head(10)['total_revenue'].plot(kind='bar')
plt.title('Total Revenue by Country (Top 10)')
plt.xlabel('Country')
plt.ylabel('Revenue (£)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## Key Findings Summary

### Customer Behavior
- Total customers: X,XXX
- Repeat customer rate: XX%
- Average orders per customer: X.X
- Average revenue per customer: £XXX

### Sales Behavior
- Total revenue: £X,XXX,XXX
- Peak month: XXXX
- Seasonal patterns observed: [Yes/No]

### Product Behavior
- Average unique products per customer: XX
- Top product category: XXX

### Cancellation Behavior
- Cancellation rate: X.X%
- Customers with cancellations: XX%
- Revenue impact: £XX,XXX (X.X% of total)

### Geographic Distribution
- Top country: United Kingdom (XX% of revenue)
- International presence: XX countries